In [65]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [66]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5" 

In [67]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
        "stop_sequences" : stop_sequences
    }

    if system:
        params["system"] = system
        
    response = client.messages.create(**params)
    # response.content can include a ThinkingBlock before the TextBlock
    # (adaptive thinking is on by default on this model), so find the
    # first text block instead of assuming content[0] is text.
    return next(block.text for block in response.content if block.type == "text")   


In [68]:
# Make a start listing of messages
messages = []


# Add in the initial user message
add_user_message(messages, "Generate a very short event bridge rule as JSON")
add_assistant_message(messages,"```json")

# Pass the message to chat() to get a response from the model
text = chat(messages, stop_sequences=["```"])


print(text)




{
  "Name": "MySimpleRule",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



In [78]:
import json
def generate_dataset():
    prompt = """ 
Generate a evaluation dataset for a prompt evaluation. 
The dataset evaluate programs that generate Python, JSON, or RegEx specifically for AWS-related tasks. 
Generate an array of JSON objects, where each object contains a "task" that requires Python, JSON, or RegEx to complete.

Example output:
```json
[
     {
        "task" : "Descriptopn of task",
        "format" : "json" or "python" or "regex"
        "solution_criteria" : "key criteria for evaluating the solution"
     },
     additional...
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a signle Regex.
* Focus on tasks that do not require writing much code.

Please generate three objects.
"""
    messages=[]
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [79]:
dataset = generate_dataset()
dataset

with open("dataset.json","w") as f:
    json.dump(dataset, f, indent=2)

In [80]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [81]:
import re
import ast
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [82]:
from statistics import mean

def run_prompt(test_case):
    """Merge the Prompt and Test case input, then returns the result"""
    prompt = f""" 
Please solve the following task:
{test_case["task"]}
* Respond only with Python, JSON, or a plain regex
* Do not add any comments or commentary or explaination
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def run_test_case(test_case):
    """calls run_prompt, then grade the result"""
    output = run_prompt(test_case)

    ### Todo Grading
    #score = 10
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2 

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

def run_eval(test_case):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results    


In [83]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)    

Average Score: 8.666666666666666


In [84]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket_name(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., 's3://my-bucket-name/path/to/object') and extract only the bucket name",
      "format": "regex",
      "solution_criteria": "The regex should correctly extract 'my-bucket-name' from various S3 URI formats, handle bucket names with hyphens and numbers, and not capture the path portion"
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the primary requirement of extracting bucket names from standard S3 URIs. However, it lacks robustness in error handling (doesn't check for None input) and doesn't validate against AWS's actual bucket naming constraints. For production use, input validation and bucket name validation according to AWS specifications would be recommended."
  },
  {
    "output": "\nimport json\nimport re\